# Fourier Transforms II

Chapter 6 of the TN23015 book: its explanation and its exercises. Run the first cell before anything else, then work down the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tn23015 import check_answers, given, load_data, md, print, row, show, slider

In the previous lecture notebook, we looked in detail at how the 1D FFT works in Python, and saw an example of using the FFT to detect a weak sinusoidal signal in a noisy dataset. 

In this lecture notebook, you will explore the application of the 1D FFT for filtering signals, and also learn about the 2D FFT and an application of it in calculating diffraction patterns.

> **Learning goals**
>
> 1. Use the FFT to filter numerical data
> 2. Interpret artifacts in your filtered data arising from the FFT
> 3. Calculate the 2D Fourier transform of 2D data
> 4. Construct 2D images of mask patterns and calculate the far-field diffraction pattern

## Background information on Filtering Signals

A common way to remove noise from your data is to apply a [Low-pass filter](https://en.wikipedia.org/wiki/Low-pass_filter).

This is equivalent to passing an electrical signal through the following [RC circuit](https://en.wikipedia.org/wiki/RC_circuit):

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/1st_Order_Lowpass_Filter_RC.svg.png)

To predict what the output voltage is for a given input voltage, we need to know the [infinite impulse response](https://en.wikipedia.org/wiki/Infinite_impulse_response), which is the response of the circuit to an input voltage that is a delta-function $\delta(t)$.
For the circuit above, the impulse response is given by (see [here](https://en.wikipedia.org/wiki/RC_circuit#Impulse_response)):

$$
h(t) = \frac{1}{\tau} e^{-t/\tau} \theta(t)
$$

where $\tau = RC$ is the RC time constant of the circuit, and $\theta(t)$ is a step function.
We can see what this looks like using this code:

In [ ]:
# Notebook code 
tau = 1
_fig, _ax = plt.subplots(figsize=(12,3))
t = np.linspace(-5,5,1000)
h = np.exp(-t/tau)*(t>=0)
_ax.plot(t,h)
_ax.set_ylabel("Impulse response h(t)")
_ax.set_xlabel("Time");

show(_ax)

Once we know the impulse response of a filter, we can then calculate the output $V_{out}(t)$ of the filter circuit for **any** given input voltage $V_{in}(t)$ using a convolution:

$$
V_{out}(t) = \int_{-\infty}^t V_{in}(s)\, h(t-s)\, ds
$$

A nice way to understand what this does is to make a plot.
The output voltage $V_{out}$ at time $t$ is a "weighted average" of all the input voltages $V_{in}(s)$ at earlier times $s < t$, where the "weighting factor" $h(t - s)$ is decaying exponentially "backwards" in time:

In [ ]:
# Notebook code 
fig, ax = plt.subplots(figsize=(12,3))
vin = np.random.normal(size=len(t))*0.05
plt.plot(t,vin,'.', color='grey', label=r"$V_{in}$")
plt.plot(t,np.exp(t)*(t<0),'r', label=r"Impulse response")
plt.fill_between(t,np.exp(t)*(t<0), alpha=0.1, color='r')
plt.xlabel("Relative Time")
ax.get_yaxis().set_visible(False)
plt.legend()

plt.title('Illustration of the convolution integral')

show(ax)

We say that this filter has a "memory" because the value of $V_{out}$ depends on the values of $V_{in}$ in the past.
We also say that it is "causal" because the value of $V_{out}$ at time $t$ does not depend on values of $V_{in}$ in the future.

_(Aside: Any real filter you make in the lab will of course be "causal", but in software you can easily make a non-causal filter if you want...)_

While it is possible to apply a low pass filter by convolving $V_{in}(t)$ with $h(t)$ using, for example, two nested `for` loops, it is far more common in practice to perform filtering in the frequency domain.

To see how this works, we first look at an important mathematical result, known as the [convolution theorem](https://en.wikipedia.org/wiki/Convolution_theorem).
Consider now a completely general function $c(x)$ given by the convolution of two functions $f(x)$ and $g(x)$: 

$$
c(x) = \int_{-\infty}^{\infty} f(y) g(x-y) dy
$$

If $\tilde f(\omega)$ and $\tilde g(\omega)$ are the Fourier transforms of $f(x)$ and $g(x)$ respectively, then the convolution theorem states that the Fourier transform $\tilde c(\omega)$ of $c(x)$ is given by: 

$$
\tilde c(\omega) = \tilde f(\omega) \ \tilde g(\omega)
$$

This is a big deal!
Why?
In the first equation above, for each point in $c(x)$ we need to calculate a full integral.
If these are all vectors of size $N$, this integral takes $N$ steps, and so calculating $c(x)$ takes order $N^2$ steps.
In the second equation, this is only **point-wise multiplication** of the two functions!
So if we work in Fourier space, calculating the Fourier transform $\tilde c(\omega)$ takes only order $N$ steps.
If $N$ is $10^6$, this is the difference between $10^6$ steps and $10^{12}$ steps, which is a big difference. If a step takes 1 ms, this is the difference between a calculation of 17 minutes or 31 years!

(Of course, we still need to take the Fourier transform and inverse Fourier transform, but thanks to Gauss' genius, this scales only like $N\log N$, so only $6\times10^6$ steps.)

To calculate $V_{out}$ of our filter, we then need to find the Fourier transform of our impulse response function $h(t)$.
For the low pass filter above, this is given by:

$$
\tilde h(\omega) = \frac{1}{1+i\omega \tau}
$$

The Fourier transform $\tilde h(\omega)$ of the impulse response function is also referred to as the [Transfer Function](https://en.wikipedia.org/wiki/Transfer_function).
In our case, it tells you how the amplitude and phase of an oscillating input signal is related to the amplitude and phase of the output signal.
Using [phasor notation](https://en.wikipedia.org/wiki/Phasor) $V(t) = \tilde V e^{i\omega t}$, the input and output signals are related by:

$$
\tilde V_{out}(\omega) = \tilde h(\omega) \tilde V_{in}(\omega)
$$

You can see a plot of the transfer function for an RC filter here:

In [ ]:
# Notebook code 
tau_2 = 1
w = np.geomspace(1e-3,1e3, 100)
h_2 = 1/(1+1j*w*tau_2)

_fig, _axs = plt.subplots(1, 2, figsize=(12,4))
_ax = _axs[0]
_ax.plot(w,np.abs(h_2))
_ax.set_xlabel(r"$\omega$")
_ax.set_ylabel(r"$|h(\omega)|$")
_ax.set_xscale('log')
_ax.set_yscale('log')
_ax.grid()

_ax = _axs[1]
_ax.plot(w, np.angle(h_2)/np.pi*180)
_ax.set_xscale('log')
_ax.set_xlabel(r"$\omega$")
_ax.set_ylabel(r"Phase angle of $\tilde h$ (degrees)")
_ax.grid()
_ax.set_yticks(np.arange(-90,1,15));

show(_fig)

To get back to the output signal $V_{out}(t)$, all we have to do is then apply the inverse Fourier transform:

$$
V_{out}(t) = \frac{1}{2\pi} \int_{-\infty}^{\infty} \tilde V_{in}(\omega) \tilde h(\omega) e^{i\omega t} d\omega
$$

In python, you would first use `np.fft.fft()` to calculate the FFT of the input signal, multiply this by the filter transfer function you want to use, and then take the inverse FFT using `np.fft.ifft()`.
Done!

## Filtering a square pulse

You will start by calculating the result of filtering a square pulse using a low-pass filter.

### Exercise 6.1

Make a square voltage pulse:  

$$
V(t) = 
\begin{cases}
0 &  \text{for } 0 <t \leq 50 \text{ s}\\
1 &  \text{for } 50 <t \leq 100 \text{ s} \\
0 &  \text{for } 100 <t \leq 150 \text{ s}
\end{cases}
$$

Your pulse should include 1000 points.

In [ ]:
# Fill these in as you work through the exercise.
t_2 = v = None

# t_2 = np.linspace(...)

# If you are well versed with vectorized comparison operators, you
# can create v in one line! Otherwise, a for loop is OK too...
# v = ....

# Do not edit or remove the boiler-plate code below.
_ax = None
if given(t_2, v):
    _fig, _ax = plt.subplots()
    _ax.plot(t_2,v)
    _ax.set_ylabel("Voltage (V)")
    _ax.set_xlabel("Time (s)")

show(_ax, t=t_2, v=v)

In [ ]:
check_answers(t_2=t_2, v=v, key="answer_8_1a")

### Exercise 6.2

Write a function to use a FFT to apply a low-pass filter to the data.
Your filter should have a time constant $\tau = 5$ seconds.
The cell includes code to make a plot of the original data along with the filtered data. 

_Note:_ When using `np.fft.ifft()`, it will return a complex-valued array even if the data has no imaginary component (as it should if you have done your filtering correctly).
For plotting, you will need to use `np.real()` to convert your array to real values before plotting (if you forget, matplotlib will give you a warning message).
You should therefore have your `low_pass` function return only the real-valued part of the filtered data.

In [ ]:
def low_pass(v, tau, dt):
    return None

# Do not edit or remove the boiler-plate code below.
_ax = vfilt = None
if given(t_2, v, low_pass):
    vfilt = low_pass(v, 5, t_2[1]-t_2[0])

    _fig, _ax = plt.subplots()
    _ax.plot(t_2,v, label="$V_{in}$")
    _ax.set_ylabel("Voltage (V)")
    _ax.set_xlabel("Time (s)")
    _ax.plot(t_2, vfilt, label="$V_{out}$");
    _ax.legend()

show(_ax, t=t_2, v=v, low_pass=low_pass)

In [ ]:
check_answers(vfilt=vfilt, key="answer_8_1b")

### Exercise 6.3

Use your function to apply a low pass filter with $\tau = 20$ seconds.
As usual, your plot should have appropriate axis labels, and a legend.

In [ ]:
# Fill these in as you work through the exercise.
_ax = vfilt_2 = None

# vfilt_2 = low_pass(....)
# _fig, _ax = plt.subplots()
# _ax.plot(....)
# .....

# Do not edit or remove the boiler-plate code below.
show(_ax, vfilt=vfilt_2)

In [ ]:
check_answers(vfilt_2=vfilt_2, key="answer_8_1c")

What is strange about your filtered function at $t=0$? What is going on?

### Exercise 6.4

We can solve the problem we saw in Exercise 6.3 by using zero-padding of the array during the Fourier transform.
Read the documentation page of [`np.fft.fft`](https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html) to learn how to do this.

Write a new version of the low pass function, `low_pass2()`, this time padding the signal you are filtering with as many points as it already contains.
If you do this correctly, it will eliminate the "boundary" effects from your FFT we saw above.
The cell includes plotting code that shows your data after filtering with the new function.

In [ ]:
def low_pass2(v, tau, dt):
    return None

# Do not edit or remove the boiler-plate code below.
_ax = vfilt2 = None
if given(t_2, v, low_pass2):
    vfilt2 = low_pass2(v, 20, t_2[1]-t_2[0])
    _fig, _ax = plt.subplots()
    _ax.plot(t_2,v, label="$V_{in}$")
    _ax.set_ylabel("Voltage (V)")
    _ax.set_xlabel("Time (s)")
    _ax.plot(t_2, vfilt2, label="$V_{out}$");
    _ax.legend()

show(_ax, t=t_2, v=v, low_pass2=low_pass2)

In [ ]:
check_answers(vfilt2=vfilt2, key="answer_8_1d")

## 2D Fourier transforms 

So far, we have been considering the Fourier transform of one-dimensional data, and thinking in particular about the case of signals varying in time, for which the Fourier transform gives a decomposition of the signal in the frequency domain. 

In 2D, it is more typical to take the Fourier transform of spatial data (for example, of an image in space).
In this case, the units of the axis of your Fourier transform are not frequency, but instead a two-dimensional [wave vector](https://en.wikipedia.org/wiki/Wave_vector), usually denoted by letter $k$, with units of inverse distance m$^{-1}$.
For a 2D FT, the wave vector is a two-dimensional vector $\bf{k}$ with vector components $k_x$ and $k_y$.

The 2D FT is used extensively in solid-state physics, and also in image analysis. 

Here, we will explore two applications of the 2D Fourier transform for the calculation of Fraunhofer diffraction patterns.

### Fraunhofer Diffraction

Fraunhofer diffraction considers the very-far field scattering of collimated light off of a planar object that is much smaller than the distance to the screen (and smaller than the diameter of the beam).
A classical example of 1D Fraunhofer diffraction is the single and double slit diffraction patterns.
Here, we will explore the Fraunhofer diffraction patterns of 2D planar masks. 

Calculating the Fraunhofer diffraction pattern is actually pretty simple: in the far field limit, the image of the light's intensity on the screen is directly given by the square magnitude of the Fourier transform of the shape of the aperture.

Note that as long as the assumptions above hold, the image of the diffraction pattern on the screen is independent of the wavelength: changing from a green to red laser pointer, for example, changes only the size / scaling of the image (the dots on the wall will be further apart for a red laser than a green one, since red light has the longer wavelength).

We will aim to create a "mask" for calculating a diffraction pattern.
Your mask should consist of an array of $(N+1)\times(N+1) = 2001\times2001$ points, corresponding to $N=2000$.

To do this, we will use the `meshgrid()` command to create 2D arrays filled with X and Y coordinates.
Have your X and Y coordinates run from -1000 to 1000 nanometers.

To understand exactly what `meshgrid()` does, the code below plots the two matrices side by side with `imshow()`.

In [ ]:
N = 2000

# Lower case letters correspond to the 1D array of coordinates in the two directions
x = np.linspace(-N/2,N/2,N+1)
y = np.linspace(-N/2,N/2,N+1)

# Upper case letters correspond to the 2D matrices containing the X or Y coordinates
X, Y = np.meshgrid(x, y)

# This we need to tell python what the range of the image plot are
extent = [x[0],x[-1], y[0], y[-1]]

# We will use the subplots command to make two plots beside each other
_fig, _axs = plt.subplots(1, 2, figsize=(10,3.5))
_ax = _axs[0]
_im1 = _ax.imshow(X,origin='lower',extent=extent)
_ax.set_xlabel("X")
_ax.set_ylabel("Y")
_fig.colorbar(_im1)
_ax = _axs[1]
_im2 = _ax.imshow(Y,origin='lower', extent=extent)
_ax.set_xlabel("X")
_ax.set_ylabel("Y")
_fig.colorbar(_im2)
_fig.tight_layout()

show(_fig)

What do you see in the plots?
What do your `X` and `Y` matrices look like?

### Exercise 6.5

Use your X and Y matrices to make a mask matrix M in which all points should be 1, except for points with a radius of $r > 50$ points from the center of the array, which should be zero.
The code will make two plots of your mask: one of the full matrix, and one zoomed into the range of a 200x200 pixel zoom at the center of the image.
The mask value of 1 will indicate where the light will be transmitted, and a mask value of 0 will indicate where the light will be blocked, and so this corresponds to the diffraction through a circular aperture.

In [ ]:
# Fill these in as you work through the exercise.
_fig = M = None

# It seems like this is magic, but using the right matrices, it does just work!
# M =  _____ < 50

exts = [x[0], x[-1], y[-1], y[0]]

# Do not edit or remove the boiler-plate code below.
M_zoom = end = exts_zoom = start = zoom = None
if given(M):
    _fig, _axs = plt.subplots(1, 2, figsize=(10,3.5))

    # The full plot
    _ax = _axs[0]
    _ax.imshow(M, extent=exts)
    _ax.set_xlabel("x (nm)")
    _ax.set_ylabel("y (nm)")

    # The zoom
    _ax = _axs[1]
    zoom = 200
    start = int(len(x)/2-zoom/2)
    end = start + zoom
    M_zoom = M[start:end,start:end]
    exts_zoom = [x[start], x[end], y[start], y[end]]
    _ax.imshow(M_zoom, extent=exts_zoom)
    _ax.set_xlabel("x")
    _ax.set_ylabel("y")

show(_fig, M=M)

In [ ]:
check_answers(M=M, key="answer_8_2b")

### Exercise 6.6

Now calculate and plot the diffraction pattern from your mask.
Plot a zoom of the central 100 × 100 pixels of the diffraction pattern.
Your axis labels on your diffraction pattern should correspond to wavenumbers with the appropriate units.
(Hint: make sure to use fftshift, which also works on 2D arrays!)

In [ ]:
# Fill these in as you work through the exercise.
diffraction_pattern = end_2 = extent_zoom = k_max = k_zoom = start_2 = None
zoom_2 = 100  # the width of the zoom, in pixels

# First, take the FFT of the mask M and turn it into a diffraction pattern
# diffraction_pattern = ....

# The zoom: the central zoom_2 pixels of the image
# start_2 = ....
# end_2 = ....

# Extents: this one we have to think about carefully!
# A good trick: what is the largest wave number k_max?
# Your full image will then run from -k_max to k_max. From this,
# you can work out the extents of the zoom.
# k_max = ....
# k_zoom = k_max * zoom_2 / len(x)
# extent_zoom = [ , , , ]

# Fill in the commented lines below, but leave the rest of this block in place.
_ax = None
if given(diffraction_pattern, end_2, extent_zoom, k_max, k_zoom, start_2, M):
    _fig, _ax = plt.subplots()
    _im1 = _ax.imshow(diffraction_pattern[start_2:end_2, start_2:end_2], extent=extent_zoom)

    # _ax.set_xlabel(...)
    # _ax.set_ylabel(...)
    _fig.colorbar(_im1)

show(
    _ax,
    diffraction_pattern=diffraction_pattern,
    end=end_2,
    extent_zoom=extent_zoom,
    k_max=k_max,
    k_zoom=k_zoom,
    start=start_2,
    M=M,
)

In [ ]:
check_answers(
    diffraction_pattern=diffraction_pattern,
    k_max=k_max,
    k_zoom=k_zoom,
    extent_zoom=extent_zoom,
    key="answer_8_2c",
)

### Exercise 6.7

Here, you will make an interactive zoom of your diffraction pattern.
Fill in the code below to
show the central `zoom.value` pixels of the image.

Two sliders let you adjust the zoom and the maximum of the colormap (in logarithmic
intervals).
They are defined in their own cell — a marimo UI element has to be *read* by a
different cell from the one that creates it, and that is what makes the image redraw by itself
as you drag.

It is quite cool to zoom out and take the vmax to the lowest value in the array, particularly if
you make the figure very big by using `plt.subplots(figsize=(10, 10))` (or some size that fills
your screen nicely).

In [ ]:
# Sliders for the zoom level and the colour scale. They are displayed with the image below.
zoom_3 = slider(20, 1000, step=10, value=1000, label="Zoom (pixels)")
v_index = slider(1, 19, step=1, value=19, label="Colour scale")
controls = row(zoom_3, v_index)

# Do not edit or remove the boiler-plate code below.
show(md("*The sliders are shown with the image below.*"))

In [ ]:
# Fill these in as you work through the exercise.
end_3 = exts_2 = k_zoom_2 = start_3 = None

# To really be able to tweak the colormap over a large range,
# we will make an array of maximum colour values that is geometrically spaced
N_2 = 20
vmax_array = np.geomspace(1e1, 6e7, N_2)
vmax = vmax_array[v_index.value]

# Show the central `zoom_3.value` pixels of the diffraction pattern.
# start_3 = ...
# end_3 = ...
# k_zoom_2 = ...
# exts_2 = ...

# Do not edit or remove the boiler-plate code below.
_ax = None
if given(end_3, exts_2, k_zoom_2, start_3):
    _fig, _ax = plt.subplots(figsize=(8, 8))
    _ax.imshow(diffraction_pattern[start_3:end_3, start_3:end_3], extent=exts_2, vmax=vmax)
    _ax.set_xlabel("k$_x$ (nm$^{-1}$)")
    _ax.set_ylabel("k$_y$ (nm$^{-1}$)")

show(controls, _ax, end=end_3, exts=exts_2, k_zoom=k_zoom_2, start=start_3)

Can you understand where the funky patterns you see come from?